# 02n - zero-shot run on all 500 train+dev docs at NATIVE resolution (thin wrapper)

Qwen3.5-4B, image only, KEYED output, prompt v2, **native resolution**: `max_pixels` 2,196,480 =
2,145 visual tokens per 1240 x 1754 page (the production resolution since the 06 sweep and GG's
2026-10-03 decision; `configs/spike_qwen35_4b_img_only_native.yaml`, which is the 1260-token config
with only `name` and `max_pixels` changed). The 1260-token notebook 02 and its runs are untouched.
The zero-shot predictions of ALL 500 docs (671 pages) are (1) the zero-shot arm of the paired
comparison at native (05n, `scripts/g4_fold.py`) and (2) the training data of the calibrator, so
every trace page carries `field_logprobs` (see `src/shipdoc/trace.py`).

All logic is in `python -m shipdoc spike --split train+dev --logprobs`; this notebook only does
Drive, secrets, clone, install and bookkeeping. `PINNED_SHA` in the parameters cell is the code
commit it clones (filled by the pin commit); just Runtime -> Run all.

Run id: `zeroshot500_qwen35_4b_img_only_native_<sha7>` (no `keyed` infix: distinct from every
1260-token run `zeroshot500_qwen35_4b_img_only_keyed_<sha7>`). 05n needs `<sha7>` of THIS pin.

Flow: setup (Drive, clone at the pin, unzip data) -> shard plan -> T4-hour / compute-unit
ESTIMATE (printed before anything runs; read it, interrupt if too high) -> install -> MANDATORY
5-doc smoke gate at native (7 assertions incl. finite logprobs for every emitted field; failure
stops the notebook) -> BATCH-SIZE BENCH at native (12 dev pages at batch 1/2/4/8: byte-identity vs
batch 1, peak VRAM, pages/hour; rule: the largest size with 100% byte-identical outputs and peak
VRAM <= 14.5 GiB; a size that runs out of memory is recorded as failed and the others go on;
skipped when `BATCH_SIZE` is an int) -> full run, RESUMABLE per doc -> completion banner.

**Why the bench matters here:** VRAM at batch 8 is UNMEASURED at native. The 1260-token batch-8
peak was 13.6 GiB; the sweep measured native only at batch 1 (9.41 GiB vs 9.14 GiB for the
control). A native batch 8 may not fit in 14.5 GiB, in which case the rule picks 4 or 2 (or 1)
and the run is slower than the estimate's low row. Every figure printed before the bench is an
ESTIMATE (UNVERIFIED on a GPU).

Parameters: `BATCH_SIZE` (None = the bench decides, an int = manual and the bench is skipped),
`SHARD` ("i/K") and `MODE` ("run", or "merge" after all K shard tabs finished), exactly as in 02;
for K > 1 start every tab with the same pin and the same batch size.

The bench result (batch size) is stored with the run (`manifest.json` `batch_size`, `bench`) and
05n / 04c read it from there: the TEST submission and every OOF run must use the same batch size.

If the session dies, open the notebook again and Run all: setup repeats, the smoke gate and the
bench are skipped (they passed; finished bench sizes are never redone) and the full run resumes
after the last document in `trace.jsonl` on Drive. See notebooks/README.md.


In [ ]:
# Parameters. PINNED_SHA = the repo commit this run is pinned to (full 40-char SHA).
PINNED_SHA = "FILL_PINNED_SHA"
RUN_NAME = "zeroshot500"
CONFIG = "qwen35_4b_img_only_native"  # KEYED, prompt v2, max_pixels 2196480 (2145 tokens/page)
SPLIT = "train+dev"  # the 500 docs of splits/zeroshot500.json come from both splits
DOCS = "splits/zeroshot500.json"
EXPECTED_DOCS = 500
EXPECTED_PAGES = 671
SMOKE_DOCS = "splits/smoke5.json"  # 5 dev docs picked by meta tags (shipdoc.smoke)
SMOKE_LIMIT = 5
BENCH_DOCS = "splits/bench12.json"  # 12 dev pages for the batch-size bench (shipdoc.bench)
BATCH_SIZE = None  # None = the bench picks it; an int = manual, the bench is skipped
SHARD = "0/1"  # "i/K": this tab runs document-level shard i of K; "0/1" = everything
MODE = "run"  # "run" = smoke, bench, this shard's run; "merge" = combine the K shard folders
USE_WANDB = False  # off by default; True logs metrics only (no images / values) to the project
WANDB_PROJECT = "shipdoc-extract-debug"  # must be a PRIVATE project; see notebooks/README.md

if not PINNED_SHA or "FILL" in PINNED_SHA:
    raise ValueError("Set PINNED_SHA in the parameters cell (the pin commit fills it).")
if MODE not in ("run", "merge"):
    raise ValueError(f"MODE must be 'run' or 'merge', got {MODE!r}")
if BATCH_SIZE is not None and (isinstance(BATCH_SIZE, bool) or not isinstance(BATCH_SIZE, int)
                               or BATCH_SIZE < 1):
    raise ValueError(f"BATCH_SIZE must be None or an int >= 1, got {BATCH_SIZE!r}")
_i, _, _k = SHARD.partition("/")
if not (_i.isdigit() and _k.isdigit() and int(_k) >= 1 and int(_i) < int(_k)):
    raise ValueError(f"SHARD must look like 'i/K' with 0 <= i < K, got {SHARD!r}")
SHARD_I, SHARD_K = int(_i), int(_k)
if MODE == "merge" and SHARD_K < 2:
    raise ValueError("MODE='merge' combines K shard folders: set SHARD to '0/K' with K >= 2.")
SHA7 = PINNED_SHA[:7]
RUN_ID = f"{RUN_NAME}_{CONFIG}_{SHA7}"  # the unsharded run (and the merge target); no 'keyed'
SHARD_RUN_ID = RUN_ID if SHARD_K == 1 else f"{RUN_ID}_shard{SHARD_I}of{SHARD_K}"  # this tab


## Account

No account check in this public build: Drive is mounted as whichever Google account you sign in
with, and the inputs must be in that account's `MyDrive/shipdoc-extract/` (see the repository
README, "Reproduce from images").


In [ ]:
import subprocess
from pathlib import Path

from google.colab import drive

DRIVE_DIR = Path("/content/drive/MyDrive/shipdoc-extract")
REQUIRED_ZIPS = ["data.zip", "assignment.zip"]
RUNS_DIR = DRIVE_DIR / "runs"

drive.mount("/content/drive")

print("Public build: any Google account; inputs are read from", DRIVE_DIR)

missing = [n for n in REQUIRED_ZIPS if not (DRIVE_DIR / n).is_file()]
if missing:
    raise FileNotFoundError(f"Missing in MyDrive/shipdoc-extract/: {', '.join(missing)}")
print("Drive inputs present:", ", ".join(REQUIRED_ZIPS))
RUNS_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = RUNS_DIR / (RUN_ID if MODE == "merge" else SHARD_RUN_ID)
print("run_id:", RUN_ID, "| mode:", MODE, "| shard:", SHARD, "| batch size:", BATCH_SIZE or "bench")
print("Drive output folder:", RUN_DIR)


In [ ]:
import os

from google.colab import userdata


def get_secret(name: str, required: bool) -> str | None:
    """Read a Colab secret without ever printing it."""
    try:
        value = userdata.get(name)
    except Exception:  # SecretNotFoundError / NotebookAccessError: both mean "not usable"
        value = None
    if required and not value:
        raise RuntimeError(
            f"Colab secret {name} is missing or notebook access is off "
            "(left sidebar -> key icon -> add it and enable Notebook access)."
        )
    return value or None


HF_TOKEN = get_secret("HF_TOKEN", required=False)
WANDB_API_KEY = get_secret("WANDB_API_KEY", required=True) if USE_WANDB else None
print("HF_TOKEN:", "set" if HF_TOKEN else "not set (optional)",
      "| W&B:", "ON" if USE_WANDB else "off")


In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "FILL_PUBLIC_REPO_URL"
REPO = Path("/content/shipdoc-extract")
if not PINNED_SHA or "FILL" in PINNED_SHA:
    raise ValueError("Set PINNED_SHA in the parameters cell to the commit this run is pinned to.")
if "FILL" in REPO_URL:
    raise ValueError("REPO_URL is a placeholder: the public repository is not published yet.")


def git(*args: str) -> str:
    """Run git in the repo dir (or /content). The repository is public: no credentials at all."""
    cwd = REPO if (REPO / ".git").is_dir() else "/content"
    res = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True, check=False)
    if res.returncode:
        raise RuntimeError(f"git {args[0]} failed (exit {res.returncode}): {res.stderr[-800:]}")
    return res.stdout.strip()


if (REPO / ".git").is_dir():
    git("fetch", "origin")
else:
    git("clone", REPO_URL, str(REPO))
git("checkout", PINNED_SHA)
head = git("rev-parse", "HEAD")
pinned_full = git("rev-parse", f"{PINNED_SHA}^{{commit}}")
assert head == pinned_full, f"HEAD {head} != PINNED_SHA {pinned_full}"
print("HEAD =", head)


In [ ]:
import json
import shutil
import zipfile

LOCAL_ZIPS = Path("/content/zips")
LOCAL_ZIPS.mkdir(exist_ok=True)
CONTENT = Path("/content")
# data.zip holds data/ and assignment.zip holds assignment/; both unzip to /content (not into the
# clone), and the CLI is pointed at them with SHIPDOC_DATA_DIR / SHIPDOC_ASSIGNMENT_DIR below.
for name in ("data.zip", "assignment.zip"):
    local = LOCAL_ZIPS / name
    if not local.is_file() or local.stat().st_size != (DRIVE_DIR / name).stat().st_size:
        shutil.copyfile(DRIVE_DIR / name, local)  # local disk: Drive reads are slow
    with zipfile.ZipFile(local) as zf:
        files = [n for n in zf.namelist() if not n.endswith("/")]
        if any(not (CONTENT / n).is_file() for n in files):  # also repairs a half-extracted tree
            zf.extractall(CONTENT)
    print(f"{name}: {len(files)} files under {CONTENT / files[0].split('/')[0]}")

DATA_DIR, ASSIGNMENT_DIR = CONTENT / "data", CONTENT / "assignment"
TRAIN_LABELS, DEV_LABELS = DATA_DIR / "train" / "labels", DATA_DIR / "dev" / "labels"
assert (ASSIGNMENT_DIR / "score.py").is_file(), "assignment.zip has no score.py (official scorer)"
doc_ids = json.loads((REPO / DOCS).read_text(encoding="utf-8"))
assert len(doc_ids) == EXPECTED_DOCS == len(set(doc_ids)), f"{DOCS}: expected {EXPECTED_DOCS} docs"
missing = [d for d in doc_ids
           if not (DATA_DIR / d.split("_")[0] / "labels" / f"{d}.json").is_file()]
assert not missing, f"{len(missing)} docs without a label file, e.g. {missing[:3]}"
print(f"data OK: {len(doc_ids)} docs with labels in train+dev")


In [ ]:
# Document-level shard plan (shipdoc.shard): docs by (pages desc, doc_id), each
# given to the shard with the fewest pages so far. The CLI recomputes the same plan from
# --shard; this cell only shows it and sets the expected counts the banner checks.
import importlib.util
import sys

_shard_path = REPO / "src" / "shipdoc" / "shard.py"
_spec = importlib.util.spec_from_file_location("shipdoc_shard", _shard_path)
shard_mod = importlib.util.module_from_spec(_spec)
sys.modules["shipdoc_shard"] = shard_mod
_spec.loader.exec_module(shard_mod)

page_counts = [
    (d, len(json.loads((DATA_DIR / d.split("_")[0] / "labels" / f"{d}.json").read_text())["pages"]))
    for d in doc_ids
]
assert sum(n for _, n in page_counts) == EXPECTED_PAGES, "page count differs from EXPECTED_PAGES"
plan = shard_mod.assign_shards(page_counts, SHARD_K)
pages_of = dict(page_counts)
for _i, docs in enumerate(plan):
    mark = "  <- this tab" if _i == SHARD_I and MODE == "run" else ""
    print(f"shard {_i}/{SHARD_K}: {len(docs)} docs, {sum(pages_of[d] for d in docs)} pages{mark}")
SHARD_EXPECTED_DOCS = len(plan[SHARD_I])
SHARD_EXPECTED_PAGES = sum(pages_of[d] for d in plan[SHARD_I])


In [ ]:
if MODE == "run":
    # T4-hour and compute-unit (CU) ESTIMATE BEFORE anything runs on the GPU: read it.
    # configs/spike_speed.json has no entry for the native config, so shipdoc.nativerun derives the
    # two bounds from MEASURED paces (02 batch 8 at 1260 tokens x the 06 sweep's native cost ratio;
    # the sweep's native batch 1). The pace of the batch size the bench picks lies between them.
    # No confirmation prompt (Run all must not block): interrupt the run yourself if it is too high.
    import importlib.util
    import subprocess
    import sys

    _ge_path = REPO / "scripts" / "gpu_estimate.py"
    _spec = importlib.util.spec_from_file_location("gpu_estimate", _ge_path)
    ge = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(ge)
    _nr_path = REPO / "src" / "shipdoc" / "nativerun.py"
    _spec = importlib.util.spec_from_file_location("shipdoc_nativerun", _nr_path)
    nr = importlib.util.module_from_spec(_spec)
    sys.modules["shipdoc_nativerun"] = nr
    _spec.loader.exec_module(nr)

    SPEED = json.loads((REPO / "configs" / "spike_speed.json").read_text(encoding="utf-8"))
    smoke_ids = json.loads((REPO / SMOKE_DOCS).read_text(encoding="utf-8"))


    def read_gpu_name():
        try:
            out = subprocess.run(
                ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                capture_output=True, text=True, timeout=30, check=False,
            ).stdout.strip().splitlines()
        except (OSError, subprocess.TimeoutExpired):
            return None
        return out[0].strip() if out else None


    GPU_NAME = read_gpu_name()
    smoke_pages = ge.count_pages(smoke_ids, DEV_LABELS)
    tab_pages = SHARD_EXPECTED_PAGES  # this tab's shard (everything when SHARD is 0/1)
    print("Runtime GPU (nvidia-smi):", GPU_NAME)
    print(nr.format_zs_estimate(SPEED, tab_pages, smoke_pages, BATCH_SIZE, GPU_NAME))
    print(f"THIS TAB: shard {SHARD} ({tab_pages} pages), batch size "
          f"{BATCH_SIZE or 'from the bench'}; with K = {SHARD_K} tabs the CU of the tabs add up "
          "(K x the T4 h above).")

else:
    print("MODE = merge: skipped (estimate)")


In [ ]:
import collections
import subprocess

PY = str(REPO / ".venv" / "bin" / "python")
ENV = {
    **os.environ,
    "SHIPDOC_PROFILE": "colab",
    "HF_HOME": "/content/hf_cache",
    "SHIPDOC_RUNS_DIR": str(RUNS_DIR),
    "SHIPDOC_DATA_DIR": str(DATA_DIR),
    "SHIPDOC_ASSIGNMENT_DIR": str(ASSIGNMENT_DIR),
    "PYTORCH_ALLOC_CONF": "expandable_segments:True",  # fewer fragmentation OOMs
    "SHIPDOC_OCR_CACHE": "/content/ocr_cache",  # unused by img_only; never read
    "SHIPDOC_TMP_DIR": "/content/tmp",
    "UV_CACHE_DIR": "/content/uv_cache",
}
if USE_WANDB:
    ENV.update(WANDB_API_KEY=WANDB_API_KEY, WANDB_PROJECT=WANDB_PROJECT, WANDB_RUN_GROUP=RUN_NAME)
if HF_TOKEN:
    ENV["HF_TOKEN"] = HF_TOKEN
Path(ENV["SHIPDOC_TMP_DIR"]).mkdir(exist_ok=True)


def run_stream(
    cmd: list[str], tail: int = 40, env: dict | None = None
) -> tuple[int, list[str]]:
    """Run a command, echo its output live, return (exit code, last `tail` lines)."""
    last: collections.deque[str] = collections.deque(maxlen=tail)
    with subprocess.Popen(
        cmd, cwd=REPO, env=env or ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    ) as proc:
        for line in proc.stdout:
            print(line, end="")
            last.append(line.rstrip("\n"))
    return proc.returncode, list(last)


if subprocess.run(["uv", "--version"], capture_output=True, check=False).returncode:
    subprocess.run(["pip", "install", "-q", "uv"], check=True)
VLM_GROUP = ["--group", "vlm"] if MODE == "run" else []  # merge mode needs no torch / model stack
rc, _ = run_stream(["uv", "sync", "--frozen", *VLM_GROUP, "--python", "3.11"])
assert rc == 0, f"uv sync failed (exit {rc})"

if MODE == "run":
    subprocess.run(["nvidia-smi"], check=False)
    PROBE = (
        "import json, importlib.metadata as m, torch;"
        "print(json.dumps({'torch': torch.__version__, 'cuda': torch.cuda.is_available(),"
        " 'arch': torch.cuda.get_arch_list(),"
        " **{p: m.version(p) for p in ('transformers', 'xgrammar')}}))"
    )
    info = json.loads(subprocess.run([PY, "-c", PROBE], env=ENV, capture_output=True, text=True,
                                     check=True).stdout.strip().splitlines()[-1])
    print(info)
    assert info["cuda"], "No CUDA device: Runtime -> Change runtime type -> T4 GPU."
    # T4 is sm_75: need a native sm_75 kernel or PTX (compute_XX with XX <= 75) it can JIT.
    ok = "sm_75" in info["arch"] or any(
        a.startswith("compute_") and int(a.split("_")[1]) <= 75 for a in info["arch"]
    )
    assert ok, f"torch build has no sm_75 support: {info['arch']}"


In [ ]:
if MODE == "run":
    # MANDATORY smoke gate (keyed format, prompt v2, logprobs: unverified on a GPU):
    # the keyed config on the 5 smoke docs, then 7 assertions on the output (shipdoc.smoke):
    # JSON validity, no truncation, row counts, quantity / total share, no all-null rows, and finite
    # field logprobs for every emitted field. A failure raises: the full run below is NOT started.
    # Resumable via a status file on Drive (a passed smoke for this run id is not repeated).
    import importlib.util
    import sys
    import time

    import yaml

    _smoke_path = REPO / "src" / "shipdoc" / "smoke.py"
    _spec = importlib.util.spec_from_file_location("shipdoc_smoke", _smoke_path)
    smoke = importlib.util.module_from_spec(_spec)
    sys.modules["shipdoc_smoke"] = smoke  # dataclasses resolves annotations through sys.modules
    _spec.loader.exec_module(smoke)

    SMOKE_RUNS = RUNS_DIR / "smoke"
    SMOKE_RUN_ID = f"smoke_{RUN_ID}"
    SMOKE_STATUS_PATH = RUNS_DIR / f"{RUN_NAME}_smoke_status.json"
    smoke_status = json.loads(SMOKE_STATUS_PATH.read_text()) if SMOKE_STATUS_PATH.is_file() else {}
    SMOKE_ENV = {**ENV, "SHIPDOC_RUNS_DIR": str(SMOKE_RUNS)}
    for _k in ("WANDB_API_KEY", "WANDB_PROJECT", "WANDB_RUN_GROUP"):
        SMOKE_ENV.pop(_k, None)  # smoke runs never log to W&B
    SMOKE_PASSED = False


    def save_smoke_status() -> None:
        tmp = SMOKE_STATUS_PATH.with_suffix(".tmp")
        tmp.write_text(json.dumps(smoke_status, indent=1))
        tmp.replace(SMOKE_STATUS_PATH)  # atomic: a disconnect never leaves a half-written file


    if smoke_status.get("state") == "passed" and smoke_status.get("run_id") == SMOKE_RUN_ID:
        SMOKE_PASSED = True
        print(f"SKIP smoke: already passed ({SMOKE_RUN_ID})")
    else:
        run_dir = SMOKE_RUNS / SMOKE_RUN_ID
        t0 = time.time()
        rc, tail, checks, warns, error = 0, [], [], [], None
        try:
            max_new = int(yaml.safe_load((REPO / f"configs/spike_{CONFIG}.yaml").read_text())[
                "max_new_tokens"])
            if not (run_dir / "metrics.json").is_file():  # a finished run is re-checked, not re-run
                print(f"=== {SMOKE_RUN_ID}: {len(smoke_ids)} docs ===", flush=True)
                cmd = [PY, "-m", "shipdoc", "spike", "--config", f"configs/spike_{CONFIG}.yaml",
                       "--docs", SMOKE_DOCS, "--split", "dev", "--run-id", SMOKE_RUN_ID, "--resume",
                       "--limit", str(SMOKE_LIMIT), "--logprobs"]
                rc, tail = run_stream(cmd, env=SMOKE_ENV)
            if rc == 0:
                checks = smoke.load_and_check(run_dir, DEV_LABELS, max_new, require_logprobs=True)
                warns = smoke.load_warnings(run_dir)  # non-blocking: never raises, never fails
        except (OSError, KeyError, ValueError) as exc:  # fail closed: could not verify = fail
            rc, error = 1, f"{type(exc).__name__}: {exc}"
        if rc != 0:
            state = "error"
            print(f"SMOKE ERROR: exit {rc} {error or ''}; last output: {tail[-3:]}")
        else:
            state = "passed" if all(c.passed for c in checks) else "failed"
            print(smoke.format_table(CONFIG, checks, warns))
        smoke_status = {
            "state": state,
            "run_id": SMOKE_RUN_ID,
            "exit_code": rc,
            "seconds": round(time.time() - t0, 1),
            "checks": [{"name": c.name, "passed": c.passed, "detail": c.detail} for c in checks],
            "warnings": [{"name": w.name, "count": w.count, "detail": w.detail} for w in warns],
            "error": error,
            "tail": tail[-5:] if rc else [],
        }
        save_smoke_status()
        SMOKE_PASSED = state == "passed"

    if not SMOKE_PASSED:
        raise RuntimeError(
            "SMOKE GATE FAILED: the full run is NOT started. GG: report back with "
            f"{SMOKE_STATUS_PATH} and the per-assertion table printed above, plus (they stay on "
            f"Drive, do not paste document values into chat) "
            f"{SMOKE_RUNS / SMOKE_RUN_ID}/trace.jsonl "
            "and predictions.json. Do not edit thresholds to make it pass."
        )
    print("SMOKE GATE PASSED: starting the full run.")

else:
    print("MODE = merge: skipped (smoke gate)")


In [ ]:
if MODE == "run":
    # NATIVE (2,145 tokens/page): VRAM at batch 8 is UNMEASURED here. A size that runs out of
    # memory is recorded as failed in bench_result.json (shipdoc.bench), the other sizes go on.
    # BATCH-SIZE BENCH (after the smoke). BATCH_SIZE = None: `python -m shipdoc bench`
    # runs the real pipeline on the 12 dev pages of splits/bench12.json at batch 1, 2, 4, 8 and
    # picks the pages per generate call by the rule printed in the banner (largest size with 100%
    # byte-identical outputs vs batch 1 and peak VRAM <= 14.5 GiB; else the near-identical fallback,
    # flagged as a DEVIATION; else 1). The result is saved on Drive and reused by a later tab / a
    # resumed session (only while code, config and model are the same). BATCH_SIZE = int: manual,
    # no bench.
    if not SMOKE_PASSED:
        raise RuntimeError(
            "The smoke gate has not passed: the bench and the full run are NOT started."
        )
    BENCH_DIR = RUNS_DIR / f"bench_{CONFIG}_{SHA7}"
    BENCH_RESULT = None  # path of the bench result the full run is sized by (None = manual)
    if BATCH_SIZE is not None:
        BATCH = BATCH_SIZE
        print("=" * 78)
        print(f"MANUAL BATCH SIZE {BATCH}: the bench is SKIPPED, nothing verifies that batch")
        print(f"{BATCH} gives the same outputs as batch 1 (the bench would). Use the same size")
        print("for every shard tab and for the dev / test runs that are compared with this one.")
        print("=" * 78)
    else:
        bench_cmd = [PY, "-m", "shipdoc", "bench", "--config", f"configs/spike_{CONFIG}.yaml",
                     "--docs", BENCH_DOCS, "--out-dir", str(BENCH_DIR), "--reuse"]
        rc, tail = run_stream(bench_cmd)
        BENCH_RESULT = BENCH_DIR / "bench_result.json"
        if rc != 0 or not BENCH_RESULT.is_file():
            raise RuntimeError(
                f"BENCH FAILED (exit {rc}); last output: {tail[-3:]}. The full run is NOT started. "
                "Run all again resumes (finished sizes are not redone); or set "
                "BATCH_SIZE = 1 (the unbatched path) to skip the bench."
            )
        bench_result = json.loads(BENCH_RESULT.read_text())
        BATCH = int(bench_result["chosen_batch_size"])
        results = bench_result.get("results", [])
        failed_sizes = [r["batch_size"] for r in results if not r.get("ok")]
        if failed_sizes:
            print(f"bench sizes that FAILED (recorded, not fatal; usually out of memory): "
                  f"{failed_sizes}; the rule chose among the others")
        if bench_result.get("deviation"):
            print("!" * 78)
            print(bench_result["deviation"]["text"])
            print("!" * 78)
    print(f"BATCH SIZE FOR THE FULL RUN: {BATCH} ({'bench' if BENCH_RESULT else 'manual'})")
    print("The TEST submission MUST use this same batch size (and the bench result "
          f"{BENCH_RESULT or 'of the dev run'}): greedy outputs are only comparable at equal "
          "batch size.")

else:
    print("MODE = merge: skipped (bench)")


In [ ]:
if MODE == "run":
    # Full run over the documents of this tab (all 500 unless SHARD is i/K), one subprocess,
    # RESUMABLE per doc: the CLI appends one line to trace.jsonl and atomically rewrites
    # predictions.json / progress.json after every document, and --resume skips the doc_ids
    # already in trace.jsonl. A disconnect loses at most the window of documents in flight (a few
    # batches; a document is traced when ALL its pages are done). The batch size is stored in
    # manifest.json on the first start and a resume reuses it. OOM, disconnect and quota handling:
    # notebooks/README.md.
    import re
    import threading
    import time
    from datetime import UTC, datetime

    if not SMOKE_PASSED:
        raise RuntimeError("The smoke gate has not passed: the full run is NOT started.")
    if "BATCH" not in globals():
        raise RuntimeError("The batch-size cell has not run: the full run is NOT started.")
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    OOM_RE = re.compile(r"out of memory|OutOfMemoryError", re.IGNORECASE)
    SESSIONS_PATH = RUN_DIR / "sessions.json"  # wall-clock log, one entry per Run all
    sessions = json.loads(SESSIONS_PATH.read_text()) if SESSIONS_PATH.is_file() else []


    def save_json(path, obj) -> None:
        tmp = path.with_suffix(".tmp")
        tmp.write_text(json.dumps(obj, indent=1))
        tmp.replace(path)  # atomic: a disconnect never leaves a half-written file


    def read_progress() -> dict:
        try:
            return json.loads((RUN_DIR / "progress.json").read_text())
        except (OSError, ValueError):
            return {}


    def watch_progress(stop: threading.Event, every: float = 60.0) -> None:
        last, t_start = -1, time.time()
        while not stop.wait(every):
            p = read_progress()
            if p.get("done") != last:
                last = p.get("done")
                print(f"[progress] {p.get('done')}/{p.get('total')} docs, "
                      f"last {p.get('last_doc')}, "
                      f"{(time.time() - t_start) / 60:.0f} min this session", flush=True)


    THIS_DOCS = globals().get("SHARD_EXPECTED_DOCS", EXPECTED_DOCS)  # docs of this tab's shard
    done_before = read_progress().get("done", 0)
    print(f"=== {SHARD_RUN_ID}: {done_before}/{THIS_DOCS} docs already done; resuming the rest "
          f"(batch size {BATCH}, shard {SHARD}) ===")
    session = {"start": datetime.now(UTC).isoformat(), "end": None, "seconds": None,
               "docs_done_at_start": done_before, "exit_code": None}
    sessions.append(session)
    save_json(SESSIONS_PATH, sessions)
    cmd = [PY, "-m", "shipdoc", "spike", "--config", f"configs/spike_{CONFIG}.yaml", "--docs", DOCS,
           "--split", SPLIT, "--run-id", RUN_ID, "--resume", "--logprobs",
           "--batch-size", str(BATCH), "--shard", SHARD]
    if BENCH_RESULT is not None:
        cmd += ["--bench-result", str(BENCH_RESULT)]  # recorded in the manifest, checked to match
    if USE_WANDB:
        cmd.append("--wandb")
    stop = threading.Event()
    threading.Thread(target=watch_progress, args=(stop,), daemon=True).start()
    t0 = time.time()
    try:
        rc, tail = run_stream(cmd)
    finally:
        stop.set()
        session.update(end=datetime.now(UTC).isoformat(), seconds=round(time.time() - t0, 1))
        save_json(SESSIONS_PATH, sessions)
    session["exit_code"] = rc
    save_json(SESSIONS_PATH, sessions)
    if rc != 0:
        oom = OOM_RE.search("\n".join(tail))
        raise RuntimeError(
            f"Full run stopped (exit {rc}{', CUDA out of memory' if oom else ''}) after "
            f"{read_progress().get('done')}/{THIS_DOCS} docs; last output: {tail[-3:]}. Everything "
            "done so far is on Drive: Run all again resumes. If it stops again at the same "
            "document "
            f"({read_progress().get('last_doc')} was the last finished), report back."
        )
    print(f"--- full run finished (exit 0) in {(time.time() - t0) / 60:.1f} min this session ---")

else:
    print("MODE = merge: skipped (full run)")


In [ ]:
# MODE = 'merge': combine the K shard folders into the unsharded run folder (CPU).
# `python -m shipdoc merge-shards` REFUSES (and writes nothing) unless every shard is complete and
# the shards agree on code SHA, config hash, model revision, seed, batch size and doc list, no
# document is twice, and the merged counts are EXPECTED_DOCS / EXPECTED_PAGES. The merged
# trace.jsonl / predictions.json / metrics.json equal an unsharded run (tests/test_shards.py).
if MODE == "merge":
    merge_cmd = [PY, "-m", "shipdoc", "merge-shards", "--config", f"configs/spike_{CONFIG}.yaml",
                 "--docs", DOCS, "--split", SPLIT, "--run-id", RUN_ID, "--shards", str(SHARD_K),
                 "--expected-docs", str(EXPECTED_DOCS), "--expected-pages", str(EXPECTED_PAGES)]
    rc, tail = run_stream(merge_cmd)
    if rc != 0:
        raise RuntimeError(f"MERGE REFUSED (exit {rc}): {tail[-3:]}. Nothing was merged; fix the "
                           "shard that is named above (rerun it with Run all) and merge again.")
    SESSIONS_PATH = RUN_DIR / "sessions.json"  # the shard sessions, tagged, for the banner
    SMOKE_STATUS_PATH = RUNS_DIR / f"{RUN_NAME}_smoke_status.json"
    smoke_status = json.loads(SMOKE_STATUS_PATH.read_text()) if SMOKE_STATUS_PATH.is_file() else {}
    SMOKE_RUNS, SMOKE_RUN_ID = RUNS_DIR / "smoke", f"smoke_{RUN_ID}"

    def read_progress() -> dict:
        try:
            return json.loads((RUN_DIR / "progress.json").read_text())
        except (OSError, ValueError):
            return {}
else:
    print("MODE = run: nothing to merge in this tab.")


In [ ]:
# Completion banner. Reads only files on Drive: also runnable alone after a resume.
import json

metrics_path = RUN_DIR / "metrics.json"
metrics = json.loads(metrics_path.read_text()) if metrics_path.is_file() else {}
progress = read_progress()
traces = [json.loads(ln) for ln in (RUN_DIR / "trace.jsonl").read_text().splitlines() if ln.strip()]
n_docs = len({t["doc_id"] for t in traces})
pages = [p for t in traces for p in t["pages"]]
with_lp = sum(isinstance(p.get("field_logprobs"), list) for p in pages)
sessions = json.loads(SESSIONS_PATH.read_text()) if SESSIONS_PATH.is_file() else []
secs = sum(s["seconds"] for s in sessions if s.get("seconds") is not None)
unfinished = sum(s.get("seconds") is None for s in sessions)
smoke_secs = smoke_status.get("seconds") or 0
model_secs = sum(p["meta"].get("latency_s") or 0 for p in pages)
manifest_path = RUN_DIR / "manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.is_file() else {}
sharded_run = MODE == "run" and SHARD_K > 1  # this folder holds only one shard
exp_docs = globals().get("SHARD_EXPECTED_DOCS", EXPECTED_DOCS) if sharded_run else EXPECTED_DOCS
exp_pages = globals().get("SHARD_EXPECTED_PAGES", EXPECTED_PAGES) if sharded_run else EXPECTED_PAGES
complete = (progress.get("status") == "complete" and n_docs == exp_docs
            and len(pages) == exp_pages and bool(metrics))


def ci(m: dict) -> str:
    return f"{m['OVERALL']:.4f} [{m['OVERALL_ci95']['lo']:.4f}, {m['OVERALL_ci95']['hi']:.4f}]"


bar = "=" * 78
print(bar)
state = "COMPLETE" if complete else "INCOMPLETE: Run all again to resume"
what = "SHARD " + SHARD if sharded_run else ("MERGED SHARDS" if MODE == "merge" else "RUN")
print(f"ZERO-SHOT 500 {what} {state}: {RUN_DIR.name}")
print(bar)
print(f"docs done  : {n_docs} / {exp_docs}")
print(f"pages done : {len(pages)} / {exp_pages}  (with field_logprobs: {with_lp})")
if manifest:
    print(f"batch size : {manifest.get('batch_size')} ({manifest.get('batch_size_source')}); "
          f"code {str(manifest.get('code_sha'))[:12]}; "
          f"config {manifest.get('config', {}).get('hash')}")
    deviation = (manifest.get("bench") or {}).get("deviation")
    if deviation:
        print(deviation["text"])
    print("The TEST submission must use this same batch size "
          "(shipdoc.runmeta.require_same_batch_size).")
if metrics.get("scored"):
    print(f"OVERALL (official scorer, train+dev labels, {metrics['n_docs']} docs): {ci(metrics)}")
    for name, m in metrics.get("per_split", {}).items():
        print(f"  {name:<6}: {m['documents']} docs, {m['n_pages']} pages  OVERALL {ci(m)}")
    print(f"JSON-valid page rate {metrics['json_validity_rate']:.4f}  "
          f"false-fill rate {metrics['false_fill_rate']:.4f}")
else:
    print("OVERALL: not available (metrics.json missing or unscored)")
print(f"wall-clock : {secs / 3600:.2f} h over {len(sessions)} session(s)"
      f"{f' ({unfinished} unfinished, not counted)' if unfinished else ''}"
      f" + smoke {smoke_secs / 60:.1f} min; model time (sum of page latencies) "
      f"{model_secs / 3600:.2f} h")
print(f"Drive folder: {RUN_DIR}")
print("GG: download these files from that folder:")
for name in ("predictions.json", "trace.jsonl", "metrics.json", "progress.json", "sessions.json"):
    path = RUN_DIR / name
    size = f"{path.stat().st_size / 1e6:8.2f} MB" if path.is_file() else "MISSING"
    print(f"  {name:<18}{size}")
print(f"smoke run (not needed): {SMOKE_RUNS / SMOKE_RUN_ID}")
if sharded_run and complete:
    print(f"NEXT: when all {SHARD_K} shard tabs are COMPLETE, set MODE = 'merge' and SHARD = "
          f"'0/{SHARD_K}' (any tab) and Run all: it builds {RUN_ID}.")
print(f"DONE {RUN_DIR.name} docs={n_docs} pages={len(pages)} complete={complete}")
